# SOM — SMOTE Model
## Self-Organizing Map with SMOTE Oversampling
### Dataset: Academic grades of Informatics students (RPL / TKJ / MM)

# Section 0 — Install Dependencies (run once)

In [ ]:
# !pip install minisom imbalanced-learn scikit-learn pandas numpy matplotlib seaborn

# Section 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, itertools, time
from minisom import MiniSom
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from collections import Counter

warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded successfully.")

# Section 2 — Load Dataset

In [ ]:
df = pd.read_csv("dataset_students.csv")

COURSE_COLS = [
    'struktur_data', 'basis_data', 'analisis_perancangan_sistem',
    'rekayasa_perangkat_lunak',
    'jaringan_komputer', 'komunikasi_data',
    'sistem_operasi', 'organisasi_arsitektur_komputer',
    'interaksi_manusia_komputer', 'desain_web', 'sistem_multimedia'
]
LABEL_COL = 'label'   # values: 'RPL', 'TKJ', 'MM'

print("Dataset loaded:", df.shape)
print(df[LABEL_COL].value_counts())

# Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Basic info
print("Shape       :", df.shape)
print("Missing vals:\n", df.isnull().sum())
print("\nClass distribution:\n", df[LABEL_COL].value_counts())
print("\nDescriptive Statistics:\n", df.describe().T)

In [ ]:
# 3.2 Class imbalance bar chart
class_counts = df[LABEL_COL].value_counts()
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(v),
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title("Class Distribution — Original Dataset", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Specialization", fontsize=11)
ax.set_ylabel("Number of Students", fontsize=11)
ax.set_ylim(0, max(class_counts.values) * 1.18)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print("Class counts:", dict(class_counts))

# Section 4 — Data Preprocessing

In [ ]:
# 4.1 Handle missing values (mode imputation per column)
for col in COURSE_COLS:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
print("Missing after imputation:", df[COURSE_COLS].isnull().sum().sum())

In [ ]:
# 4.2 Remove duplicates
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows before: {before}  |  after dedup: {len(df)}")

In [ ]:
# 4.3 Min-Max Normalization
scaler = MinMaxScaler()
df[COURSE_COLS] = scaler.fit_transform(df[COURSE_COLS])

print("Normalization range check:")
print(df[COURSE_COLS].describe().loc[['min','max']].T)

In [ ]:
# 4.4 Distribution after normalization
flat_vals = df[COURSE_COLS].values.flatten()
bins = [(0.00, 0.25), (0.26, 0.50), (0.51, 0.75), (0.76, 1.00)]
print("Normalized value distribution:")
for lo, hi in bins:
    n = ((flat_vals >= lo) & (flat_vals <= hi)).sum()
    pct = n / len(flat_vals) * 100
    print(f"  {lo:.2f}–{hi:.2f} : {n:4d}  ({pct:.2f}%)")

In [ ]:
# 4.5 Feature Engineering: group courses into 3 specialization features
RPL_COURSES = ['analisis_perancangan_sistem', 'basis_data',
               'rekayasa_perangkat_lunak', 'struktur_data']
TKJ_COURSES = ['jaringan_komputer', 'komunikasi_data',
               'sistem_operasi', 'organisasi_arsitektur_komputer']
MM_COURSES  = ['interaksi_manusia_komputer', 'desain_web', 'sistem_multimedia']

df['feat_RPL'] = df[RPL_COURSES].mean(axis=1)
df['feat_TKJ'] = df[TKJ_COURSES].mean(axis=1)
df['feat_MM']  = df[MM_COURSES].mean(axis=1)

FEATURE_COLS = ['feat_RPL', 'feat_TKJ', 'feat_MM']
print("Aggregated features (head):")
df[FEATURE_COLS + [LABEL_COL]].head()

In [ ]:
# 4.6 Cubic transformation (feature exaggeration)
for col in FEATURE_COLS:
    df[col] = df[col] ** 3

print("Post-cubic transformation stats:")
df[FEATURE_COLS].describe().T

In [ ]:
# 4.7 Train-test split (80/20, stratified)
X = df[FEATURE_COLS].values
y = df[LABEL_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print("Train class dist:", Counter(y_train))
print("Test  class dist:", Counter(y_test))

# Section 4B — SMOTE Oversampling (applied to training set only)

In [ ]:
# Applying SMOTE only on training data prevents data leakage.
# The test set always reflects the real-world class distribution.
print("Before SMOTE:", Counter(y_train))
smote = SMOTE(random_state=42)
X_train_s, y_train_s = smote.fit_resample(X_train, y_train)
print("After  SMOTE:", Counter(y_train_s))

In [ ]:
# SMOTE before vs after class distribution chart
labels_order = ['RPL', 'TKJ', 'MM']
before = Counter(y_train)
after  = Counter(y_train_s)
x = np.arange(len(labels_order))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - width/2, [before[k] for k in labels_order], width,
            label='Before SMOTE', color='#4C72B0', alpha=0.8)
b2 = ax.bar(x + width/2, [after[k]  for k in labels_order], width,
            label='After SMOTE',  color='#55A868', alpha=0.8)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(labels_order, fontsize=11)
ax.set_ylabel("Number of Samples")
ax.set_title("Class Distribution Before vs After SMOTE", fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("smote_distribution.png", dpi=150)
plt.show()

# Section 5 — Hyperparameter Optimization via Grid Search

In [ ]:
# 5.1 Define search space (960 combinations)
param_grid = {
    'map_size'            : [(15,15),(15,20),(15,25),(20,15),(20,20),(20,25),
                              (25,10),(25,15),(25,20),(25,25)],
    'sigma'               : [1.5, 3.0],
    'learning_rate'       : [0.1, 0.3, 0.5],
    'n_iter'              : [3000, 5000, 10000, 15000],
    'neighborhood_function': ['triangle', 'gaussian'],
    'random_seed'         : [4, 42]
}

keys   = list(param_grid.keys())
values = list(param_grid.values())
combos = list(itertools.product(*values))
print(f"Total combinations: {len(combos)}")

In [ ]:
# 5.2 Helper functions: quantization error, topographic error, label neurons, predict
def quantization_error(som, data):
    return np.mean([np.linalg.norm(x - som.get_weights()[som.winner(x)])
                    for x in data])

def topographic_error(som, data):
    errors = 0
    for x in data:
        bmu1 = som.winner(x)
        dists = {(r, c): np.linalg.norm(x - som.get_weights()[r, c])
                 for r in range(som._weights.shape[0])
                 for c in range(som._weights.shape[1])}
        sorted_d = sorted(dists, key=dists.get)
        bmu2 = sorted_d[1]
        if max(abs(bmu1[0]-bmu2[0]), abs(bmu1[1]-bmu2[1])) > 1:
            errors += 1
    return errors / len(data)

def label_neurons(som, X_tr, y_tr):
    rows, cols, _ = som.get_weights().shape
    counts = {(r,c): Counter() for r in range(rows) for c in range(cols)}
    for x, lbl in zip(X_tr, y_tr):
        w = som.winner(x)
        counts[w][lbl] += 1
    labels = {}
    for pos, cnt in counts.items():
        labels[pos] = cnt.most_common(1)[0][0] if cnt else None
    for pos in [p for p, l in labels.items() if l is None]:
        r, c = pos
        for d in range(1, max(rows, cols)):
            neighbours = [(r+dr, c+dc) for dr in range(-d,d+1) for dc in range(-d,d+1)
                          if (abs(dr)==d or abs(dc)==d) and 0<=r+dr<rows and 0<=c+dc<cols]
            labelled = [labels[n] for n in neighbours if labels.get(n) is not None]
            if labelled:
                labels[pos] = Counter(labelled).most_common(1)[0][0]
                break
    return labels

def predict(som, labels, X):
    return [labels[som.winner(x)] for x in X]

print("Helper functions defined.")

In [ ]:
# 5.3 Run grid search on SMOTE data
print("Starting grid search — this may take several minutes...")
t0 = time.time()

best_acc_s  = -1
best_params_s = None
results_s = []

for combo in combos:
    params = dict(zip(keys, combo))
    r, c = params['map_size']
    som = MiniSom(
        x=r, y=c,
        input_len=X_train_s.shape[1],
        sigma=params['sigma'],
        learning_rate=params['learning_rate'],
        neighborhood_function=params['neighborhood_function'],
        random_seed=params['random_seed']
    )
    som.random_weights_init(X_train_s)
    som.train_random(X_train_s, params['n_iter'], verbose=False)

    lbl_map = label_neurons(som, X_train_s, y_train_s)
    y_pred  = predict(som, lbl_map, X_test)

    acc = accuracy_score(y_test, y_pred)
    qe  = quantization_error(som, X_train_s)
    results_s.append({**params, 'accuracy': acc, 'QE': qe})

    if acc > best_acc_s:
        best_acc_s    = acc
        best_params_s = params

elapsed = time.time() - t0
print(f"Grid search done in {elapsed:.1f}s")
print(f"Best accuracy : {best_acc_s:.4f}")
print(f"Best params   : {best_params_s}")

gs_s_df = pd.DataFrame(results_s)
gs_s_df.to_csv("gs_smote_results.csv", index=False)
print("Results saved to gs_smote_results.csv")

In [ ]:
# 5.4 Top-10 configurations
top10 = gs_s_df.sort_values('accuracy', ascending=False).head(10)
display(top10)

# Section 6 — Train Final SMOTE SOM Model

In [ ]:
# Re-train with best configuration for reproducibility
bp_s = best_params_s
r, c = bp_s['map_size']
som_smote = MiniSom(
    x=r, y=c,
    input_len=X_train_s.shape[1],
    sigma=bp_s['sigma'],
    learning_rate=bp_s['learning_rate'],
    neighborhood_function=bp_s['neighborhood_function'],
    random_seed=bp_s['random_seed']
)
som_smote.random_weights_init(X_train_s)
som_smote.train_random(X_train_s, bp_s['n_iter'], verbose=True)

lbl_smote    = label_neurons(som_smote, X_train_s, y_train_s)
y_pred_smote = predict(som_smote, lbl_smote, X_test)

print("\nFinal SMOTE model trained.")
print(f"Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}")

# Section 7 — Model Evaluation

In [ ]:
# 7.1 Map Quality: QE and TE
qe_s = quantization_error(som_smote, X_train_s)
te_s = topographic_error(som_smote, X_train_s)
print(f"Quantization Error (QE) : {qe_s:.6f}")
print(f"Topographic Error  (TE) : {te_s:.4f}")

In [ ]:
# 7.2 Classification metrics
print("Classification Report:")
print(classification_report(y_test, y_pred_smote, target_names=labels_order, digits=4))

In [ ]:
# 7.3 Confusion matrix heatmap
color_map = {'RPL': '#4C72B0', 'TKJ': '#DD8452', 'MM': '#55A868'}
cm_s = confusion_matrix(y_test, y_pred_smote, labels=labels_order)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_order, yticklabels=labels_order,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title("Confusion Matrix — SMOTE Model", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.savefig("cm_smote.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7.4 Per-class metrics bar chart
report_s = classification_report(y_test, y_pred_smote, target_names=labels_order, output_dict=True)
metrics  = ['precision', 'recall', 'f1-score']
x = np.arange(len(labels_order))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, m in enumerate(metrics):
    vals = [report_s[cls][m] for cls in labels_order]
    bars = ax.bar(x + i*width, vals, width, label=m.capitalize(), alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + width)
ax.set_xticklabels(labels_order, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Per-class Metrics — SMOTE Model", fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("metrics_smote.png", dpi=150, bbox_inches='tight')
plt.show()

# Section 8 — SOM Map Visualizations

In [ ]:
# 8.1 U-Matrix
umat_s = som_smote.distance_map()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.pcolormesh(umat_s.T, cmap='bone_r')
plt.colorbar(im, ax=ax, label='Distance')
ax.set_title("U-Matrix — SMOTE Model", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Neuron X")
ax.set_ylabel("Neuron Y")
plt.tight_layout()
plt.savefig("umatrix_smote.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.2 Classification map (neuron labels)
r_map, c_map = bp_s['map_size']

fig, ax = plt.subplots(figsize=(10, 8))
for ri in range(r_map):
    for ci in range(c_map):
        lbl = lbl_smote.get((ri, ci))
        col = color_map.get(lbl, '#cccccc')
        ax.add_patch(plt.Rectangle([ci, ri], 1, 1, color=col, alpha=0.75))

for x_pt, lbl in zip(X_train_s, y_train_s):
    bmu = som_smote.winner(x_pt)
    ax.plot(bmu[1]+0.5, bmu[0]+0.5, 'o', color=color_map[lbl],
            markeredgecolor='white', markersize=5, markeredgewidth=0.5)

patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=patches, loc='upper right', fontsize=10)
ax.set_xlim(0, c_map)
ax.set_ylim(0, r_map)
ax.set_title("SOM Classification Map — SMOTE", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Neuron Column")
ax.set_ylabel("Neuron Row")
plt.tight_layout()
plt.savefig("som_map_smote.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.3 Average feature values per specialization
feat_names = ['RPL Feature', 'TKJ Feature', 'MM Feature']

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for i, (cls, col) in enumerate(color_map.items()):
    mask = y_train_s == cls
    vals = X_train_s[mask].mean(axis=0)
    axes[i].bar(feat_names, vals, color=col, alpha=0.8, edgecolor='white')
    axes[i].set_title(f"Specialization: {cls}", fontsize=12, fontweight='bold')
    axes[i].set_ylim(0, max(X_train_s.max(), 0.01) * 1.1)
    for j, v in enumerate(vals):
        axes[i].text(j, v + 0.002, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    axes[i].spines[['top', 'right']].set_visible(False)
fig.suptitle("Average Feature Values per Specialization — SMOTE",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("feature_profile_smote.png", dpi=150, bbox_inches='tight')
plt.show()

# Section 9 — Baseline vs SMOTE Comparison

In [ ]:
# Paste baseline values here from Baseline notebook output
# OR run both notebooks and use the exported CSV files
BASELINE_ACC    = 0.79   # <-- replace with actual value
BASELINE_F1     = 0.76   # <-- replace with actual value
BASELINE_RECALL = {'RPL': 0.85, 'TKJ': 0.67, 'MM': 0.62}  # <-- replace

smote_acc    = accuracy_score(y_test, y_pred_smote)
smote_f1     = f1_score(y_test, y_pred_smote, average='macro')
smote_recall = {cls: report_s[cls]['recall'] for cls in labels_order}

In [ ]:
# 9.1 Accuracy & Macro F1 comparison bar chart
x = np.arange(2)
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - width/2, [BASELINE_ACC, smote_acc], width, label='Accuracy', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x + width/2, [BASELINE_F1,  smote_f1],  width, label='Macro F1', color='#55A868', alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Baseline', 'SMOTE'], fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Baseline vs SMOTE — Accuracy & Macro F1", fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("comparison_acc_f1.png", dpi=150)
plt.show()

In [ ]:
# 9.2 Per-class recall comparison (sensitivity analysis)
x = np.arange(len(labels_order))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, [BASELINE_RECALL[k] for k in labels_order], width,
       label='Baseline', color='#4C72B0', alpha=0.85)
ax.bar(x + width/2, [smote_recall[k]    for k in labels_order], width,
       label='SMOTE',    color='#55A868', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels_order, fontsize=12)
ax.set_ylim(0, 1.25)
ax.set_ylabel("Recall")
ax.set_title("Per-class Recall: Baseline vs SMOTE (Sensitivity Analysis)",
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig("comparison_recall.png", dpi=150)
plt.show()

# Section 10 — Summary Table

In [ ]:
summary = {
    'Model'                : 'SMOTE',
    'Map Size'             : f"{bp_s['map_size'][0]}×{bp_s['map_size'][1]}",
    'Sigma'                : bp_s['sigma'],
    'Learning Rate'        : bp_s['learning_rate'],
    'Iterations'           : bp_s['n_iter'],
    'Neighborhood Function': bp_s['neighborhood_function'],
    'QE'                   : round(qe_s, 6),
    'TE'                   : round(te_s, 4),
    'Accuracy'             : round(accuracy_score(y_test, y_pred_smote), 4),
    'Macro Precision'      : round(precision_score(y_test, y_pred_smote, average='macro'), 4),
    'Macro Recall'         : round(recall_score(y_test, y_pred_smote, average='macro'), 4),
    'Macro F1'             : round(f1_score(y_test, y_pred_smote, average='macro'), 4),
}
pd.DataFrame([summary]).T.rename(columns={0: 'Value'})